In [1]:
import pandas as pd 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report


x_train = pd.read_csv("modeling_data/x_train.csv")
x_test = pd.read_csv("modeling_data/x_test.csv")
y_train = pd.read_csv("modeling_data/y_train.csv").squeeze()
y_test = pd.read_csv("modeling_data/y_test.csv").squeeze()


print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(248, 5)
(62, 5)
(248,)
(62,)


In [2]:
scaler = StandardScaler()

x_train_scaled= scaler.fit_transform(x_train)
x_test_scaled= scaler.fit_transform(x_test)

Logistic_model = LogisticRegression()

Logistic_model.fit(x_train_scaled, y_train)
y_pred_logistic = Logistic_model.predict(x_test_scaled)

Logistic_model.coef_

array([[ 0.71374503, -0.2480162 , -0.71147408, -0.99593409,  3.20362342]])

بخش 1:

evaluate results(ارزیابی نتایج نسبت به اهداف کسب و کار)

با توجه به اینکه من دقت بالای 80 درصد مخواستم حدود 87 درصد بهم دقت داد این مدل 

ولی چون داده پزشکی هستش و دقت اهمیت خییلی زیادی داره و recall 88 درصد درست انجام میده باید سعی کنم بهترش کنم

In [3]:
Logistic_model_balanced = LogisticRegression(class_weight="balanced")
Logistic_model_balanced.fit(x_train_scaled, y_train)
y_pred_balanced = Logistic_model_balanced.predict(x_test_scaled)

print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.76      0.95      0.84        20
           1       0.97      0.86      0.91        42

    accuracy                           0.89        62
   macro avg       0.87      0.90      0.88        62
weighted avg       0.90      0.89      0.89        62



برعکس انتظارم این کد دقت رو پایین تر اورد رو داده های انرمال پس باید روش های دیگه رو تست کنم

In [4]:
y_proba = Logistic_model.predict_proba(x_test_scaled)
y_proba[:5]

array([[0.31579128, 0.68420872],
       [0.75999021, 0.24000979],
       [0.1422249 , 0.8577751 ],
       [0.57372957, 0.42627043],
       [0.75147493, 0.24852507]])

In [5]:
y_proba_abnormal = y_proba[: , 1]

threshold = 0.3
y_pred_threshold = (y_proba_abnormal >= threshold).astype(int)

print(classification_report(y_test, y_pred_threshold))


              precision    recall  f1-score   support

           0       1.00      0.45      0.62        20
           1       0.79      1.00      0.88        42

    accuracy                           0.82        62
   macro avg       0.90      0.72      0.75        62
weighted avg       0.86      0.82      0.80        62



تو انرمال بودن recall 1 شد هیچ بیماری از دستمون در نرفت 

ولی نرمال شد 0.45 یعنی خیلی از افراد سالم رو هم مریض در نظر گرفته پس این مناسب نیست

In [6]:
y_proba_abnormal = y_proba[: , 1]

threshold = 0.4
y_pred_threshold04 = (y_proba_abnormal >= threshold).astype(int)

print(classification_report(y_test, y_pred_threshold04))


              precision    recall  f1-score   support

           0       0.87      0.65      0.74        20
           1       0.85      0.95      0.90        42

    accuracy                           0.85        62
   macro avg       0.86      0.80      0.82        62
weighted avg       0.86      0.85      0.85        62



In [7]:
y_proba_abnormal = y_proba[: , 1]

threshold = 0.5
y_pred_threshold5 = (y_proba_abnormal >= threshold).astype(int)

print(classification_report(y_test, y_pred_threshold5))


              precision    recall  f1-score   support

           0       0.77      0.85      0.81        20
           1       0.93      0.88      0.90        42

    accuracy                           0.87        62
   macro avg       0.85      0.87      0.86        62
weighted avg       0.88      0.87      0.87        62



threshold 0.4 از همه بهتر بود 
 چون 0.5 که همون پیشفرض هستش 

 0.3 ریسک اینکه افراد نرمال رو بیمار نشون بده زیاده

 ولی 0.4 از همه بهتر 

 دقت بالای 80 درصد داره 

 recall نرمال 65 و بیمار 95 هستش که برای هر دو قابل قبوله نسبت به بقیه 

 presion, f1 خیلی خوبی داره هم برای بیمار هم نرمال 

 پس انتخاب من 0.4 هستشچون بهترین تعادله 

بخش 2:

review process(مرور کل فرایند)

با توجه به اینکه رکورد هام خیلی کم بود که برای تست من خیلی داده محدودی داشتم

همین داده هم باعث شد عمق drsicion tre 10 بشه که نشون میداد اصلا مناسب داده خیلی کم نیست 

فیلد هدفم توازن نداشت و بنظرم 100 تا نمونه نرمال برای تست و یادگیری مدل کافی نیست

و همچنین داده دستی زیاد داشت دیتا ستم که مجبور شدم حذفشون کنم 

تو بخش capping یک اشتباه باعث شد کل داده های بزرگ تر از مرزم رو حدف کنم تو یکی از ستون ها که همون باعث شد برگردم ی بخشی رو عقب و دوباره انجام بدم

بخش اخر :

determine next  steps

از نظر من برای توسعه این پروژه نیازمند دیتاست خیلی بزرگ ترم تا بتونه درست پیشبینی کنه 

خودم از دقت مدلم زیاد راضی نیستم چون هدف پزشکی داره و کوچکترین اشتباهی میتونه اینده خیلی بدی داشته باشه 

همچنین دیتا ست بزرگ تر میتونه پایداری بیشتری به پروژه بده 

و حتی امکان داره حذف نکردن و کار کردن با کل ستون های دیتاست به نفعمون باشه که من اینکارو بخاطر شکی که بهش داشتم نکردم
